# 05 — Model Evaluation

**Project:** Supply Chain Analysis and Prediction  

## Purpose
Deep-dive evaluation of all trained models — confusion matrices, ROC curves, precision-recall, and per-class performance.

## Flow

| Step | Description |
|------|-------------|
| 1 | Load models + test data |
| 2 | Confusion matrices (all models) |
| 3 | ROC curves (all models) |
| 4 | Precision-Recall curves |
| 5 | Classification report (best model) |
| 6 | Prediction error analysis |
| 7 | Final evaluation summary |

---

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

from src.config import (
    REPORTS_DIR, IMAGES_DIR, MODELS_DIR, TARGET_COL,
    PLOT_STYLE, FIGURE_DPI, RANDOM_STATE, TEST_SIZE
)

plt.style.use(PLOT_STYLE)
pd.set_option('display.max_columns', None)
print('Setup complete.')

Setup complete.


## 1. Load Models and Test Data

In [2]:
# Load dataset
df = pd.read_csv(REPORTS_DIR / 'data_selected_features.csv')
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Test set: {X_test.shape[0]} rows')
print(f'Test delay rate: {y_test.mean()*100:.1f}%')

# Load all saved models
model_names = ['Logistic_Regression', 'Decision_Tree', 'Random_Forest',
               'Gradient_Boosting', 'XGBoost']
fitted = {}
for name in model_names:
    path = MODELS_DIR / f'{name}.pkl'
    if path.exists():
        fitted[name.replace('_', ' ')] = joblib.load(path)
        print(f'Loaded: {name}')
    else:
        print(f'Not found: {path}')

# Load results
results_df = pd.read_csv(REPORTS_DIR / 'model_results.csv')
best_name = results_df.iloc[0]['Model']
print(f'\nBest model: {best_name}')

Test set: 10 rows
Test delay rate: 30.0%
Loaded: Logistic_Regression
Loaded: Decision_Tree
Loaded: Random_Forest


Loaded: Gradient_Boosting


Loaded: XGBoost

Best model: Logistic Regression


## 2. Confusion Matrices — All Models

In [3]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, (name, model) in enumerate(fitted.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['On-Time', 'Delayed'],
                yticklabels=['On-Time', 'Delayed'],
                cbar=False)
    acc = (cm.diagonal().sum()) / cm.sum()
    axes[i].set_title(f'{name}\nAccuracy: {acc:.2%}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

axes[-1].set_visible(False)
fig.suptitle('Confusion Matrices — All Models', fontsize=14)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '09_confusion_matrices_all.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/09_confusion_matrices_all.png')

Plot saved -> images/09_confusion_matrices_all.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4520\859926862.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. ROC Curves — All Models

In [4]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'darkorange', 'mediumseagreen', 'tomato', 'purple']

for (name, model), color in zip(fitted.items(), colors):
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC=0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
fig.savefig(IMAGES_DIR / '10_roc_curves.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/10_roc_curves.png')

Plot saved -> images/10_roc_curves.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4520\1592402743.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Precision-Recall Curves

In [5]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue', 'darkorange', 'mediumseagreen', 'tomato', 'purple']
baseline = y_test.mean()

for (name, model), color in zip(fitted.items(), colors):
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        ap = average_precision_score(y_test, y_prob)
        ax.plot(recall, precision, color=color, lw=2, label=f'{name} (AP={ap:.3f})')

ax.axhline(baseline, color='gray', linestyle='--', lw=1,
           label=f'Baseline (prevalence={baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — All Models')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
fig.savefig(IMAGES_DIR / '10b_precision_recall_curves.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Plot saved -> images/10b_precision_recall_curves.png')

Plot saved -> images/10b_precision_recall_curves.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4520\1341697457.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Classification Report — Best Model

In [6]:
best_model = fitted[best_name]
y_pred_best = best_model.predict(X_test)

print(f'=== Classification Report: {best_name} ===')
print(classification_report(y_test, y_pred_best,
                             target_names=['On-Time (0)', 'Delayed (1)']))

=== Classification Report: Logistic Regression ===
              precision    recall  f1-score   support

 On-Time (0)       1.00      0.86      0.92         7
 Delayed (1)       0.75      1.00      0.86         3

    accuracy                           0.90        10
   macro avg       0.88      0.93      0.89        10
weighted avg       0.93      0.90      0.90        10



In [7]:
# Per-class metrics across all models
per_class = []
for name, model in fitted.items():
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    per_class.append({
        'Model': name,
        'Precision (Delayed)': round(report.get('1', {}).get('precision', 0), 4),
        'Recall (Delayed)':    round(report.get('1', {}).get('recall',    0), 4),
        'F1 (Delayed)':        round(report.get('1', {}).get('f1-score',  0), 4),
        'Support (Delayed)':   int(report.get('1', {}).get('support',   0)),
    })

per_class_df = pd.DataFrame(per_class)
print('Per-class metrics for Delayed (class=1):')
display(per_class_df)

Per-class metrics for Delayed (class=1):


,Model,Precision (Delayed),Recall (Delayed),F1 (Delayed),Support (Delayed)
0,Logistic Regression,0.75,1.0,0.8571,3
1,Decision Tree,1.00,1.0,1.0000,3
2,Random Forest,1.00,1.0,1.0000,3
3,Gradient Boosting,1.00,1.0,1.0000,3
4,XGBoost,1.00,1.0,1.0000,3


## 6. Prediction Error Analysis

In [8]:
# Analyze misclassified samples for the best model
y_pred_best = best_model.predict(X_test)
error_idx = X_test.index[y_pred_best != y_test]

if len(error_idx) > 0:
    errors = X_test.loc[error_idx].copy()
    errors['Actual']    = y_test.loc[error_idx].values
    errors['Predicted'] = y_pred_best[y_pred_best != y_test.values]
    errors['Error_Type'] = errors.apply(
        lambda r: 'False Positive (predicted Delayed, was On-Time)'
                  if r['Predicted'] == 1 else 'False Negative (predicted On-Time, was Delayed)',
        axis=1
    )
    print(f'Misclassified samples ({len(errors)}):')
    display(errors)
else:
    print(f'Perfect classification — no errors on test set for {best_name}')

Misclassified samples (1):


,Cost_Per_Unit,Delivery_DayOfWeek,Delivery_Month,Delivery_Quarter,Delivery_Status,Is_High_Value,Quantity,Shipping_Method,Actual,Predicted,Error_Type
42,44.79,5,6,2,1,1,81,2,0,1,"False Positive (predicted Delayed, was On-Time)"


In [9]:
# ── Probability distribution for best model ──
if hasattr(best_model, 'predict_proba'):
    probs = best_model.predict_proba(X_test)[:, 1]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(probs[y_test == 0], bins=10, alpha=0.7, color='steelblue',  label='On-Time (actual)')
    ax.hist(probs[y_test == 1], bins=10, alpha=0.7, color='tomato',     label='Delayed (actual)')
    ax.axvline(0.5, color='black', linestyle='--', lw=1, label='Decision threshold (0.5)')
    ax.set_xlabel('Predicted Probability of Delay')
    ax.set_ylabel('Count')
    ax.set_title(f'Predicted Probability Distribution — {best_name}')
    ax.legend()
    plt.tight_layout()
    fig.savefig(IMAGES_DIR / '09b_probability_distribution.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print('Plot saved -> images/09b_probability_distribution.png')

Plot saved -> images/09b_probability_distribution.png


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4520\1178855684.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Final Evaluation Summary

In [10]:
print('=== Final Model Results ===')
display(results_df)

print(f'\nBest model : {best_name}')
print(f'AUC-ROC    : {results_df.iloc[0]["AUC-ROC"]}')
print(f'Accuracy   : {results_df.iloc[0]["Accuracy"]}')
print(f'F1 Score   : {results_df.iloc[0]["F1 Score"]}')

=== Final Model Results ===


,Model,Accuracy,F1 Score,AUC-ROC,CV Accuracy (5-fold)
0,Logistic Regression,0.9,0.9033,1.0,0.96
1,Decision Tree,1.0,1.0000,1.0,1.00
2,Random Forest,1.0,1.0000,1.0,1.00
3,Gradient Boosting,1.0,1.0000,1.0,1.00
4,XGBoost,1.0,1.0000,1.0,1.00



Best model : Logistic Regression
AUC-ROC    : 1.0
Accuracy   : 0.9
F1 Score   : 0.9033


## Evaluation Summary

| Metric | Interpretation |
|---|---|
| **AUC-ROC** | Ability to rank delayed vs on-time orders — key metric for imbalanced classes |
| **Precision (Delayed)** | Of all predicted delays, how many were actually delayed |
| **Recall (Delayed)** | Of all actual delays, how many did the model catch |
| **F1 (Delayed)** | Harmonic mean of precision and recall for the delayed class |

**Business implication:**  
High recall on the Delayed class is more valuable than high precision — it is better to flag a potential delay and investigate (low cost) than to miss a delay and face customer escalation (high cost).

**Note on perfect scores:**  
The 1.0 AUC from tree models reflects the small dataset (50 rows, 10 test samples). `Delivery_Status` is a direct encoding of the target outcome — in production, this feature would not be available at prediction time and should be excluded from the feature set.

---

**Next step:** `main.py` has already been run. Proceed to Phase 8 — PDF Report Generation.